# Mean–CVaR portfolio optimization from one-minute S&P-style returns

**CPU-only counterpart of an NVIDIA cuFOLIO notebook.** The matching unmodified GPU notebook is in `../upstream_notebooks/`. This version uses deterministic synthetic one-minute bars so it runs on GitHub Actions without NVIDIA infrastructure.

Minute log returns are summed within each regular session. The optimizer receives daily simple returns and solves a long-only, fully invested Mean–CVaR problem on CPU.

In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src").exists():
        sys.path.insert(0, str(candidate / "src"))
        break

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cufolio_cpu.returns import daily_returns_from_minute_bars
from cufolio_cpu.synthetic import synthetic_minute_bars

In [ ]:
bars = synthetic_minute_bars(sessions=45, seed=42)
daily_log, daily_simple = daily_returns_from_minute_bars(bars)
print(f"{len(bars):,} minute bars -> {daily_simple.shape[0]} daily sessions x {daily_simple.shape[1]} assets")
daily_simple.tail()

In [ ]:
from cufolio_cpu.optimize import mean_cvar_weights

result = mean_cvar_weights(
    daily_simple, risk_aversion=5.0, confidence=0.95, max_weight=0.30
)
print("Solver status:", result.status)
print("Expected daily return:", f"{result.expected_return:.5%}")
print("Historical 95% CVaR loss:", f"{result.cvar:.5%}")
result.weights.sort_values(ascending=False)

In [ ]:
portfolio = daily_simple @ result.weights
plt.figure(figsize=(8, 3))
plt.plot((1 + portfolio).cumprod(), label="Mean–CVaR")
plt.title("Illustrative CPU Mean–CVaR cumulative return")
plt.ylabel("Growth of $1")
plt.legend()
plt.show()